# Morales2018

https://doi.org/10.1111/pce.13119

This notebook demonstrates example analyses using the Morales et al. (2018) dynamic photosynthesis model implemented in `mxlmodels`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from mxlpy import Simulator, make_protocol
from mxlmodels import get_morales2018

## Load model

In [ ]:
model = get_morales2018()
model

## Figure 5-style analysis: NPQ components

Morales et al. (2018) decomposed non-photochemical quenching into components associated with energy-dependent quenching (`qE`), chloroplast movement (`qM`), and photoinhibition (`qI`). Here, the model is simulated during a transition from low to high irradiance.

In [ ]:
model = get_morales2018()

sim = Simulator(model)
sim.simulate_protocol(
    protocol=make_protocol(
        [
            (300, {"PAR": 50}),
            (3600, {"PAR": 1000}),
        ]
    ),
    time_points_per_step=500,
)

res = sim.get_result().unwrap_or_err().get_combined()
res.head()

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(8, 6))

variables = ["NPQ", "qE", "qM", "qI"]

for ax, var in zip(axs.flat, variables):
    if var in res.columns:
        y = res[var]
    elif f"obs_{var}" in res.columns:
        y = res[f"obs_{var}"]
    else:
        continue

    ax.plot(res.index / 60, y)
    ax.set_title(var)
    ax.set_xlabel("Time / min")
    ax.set_ylabel(var)

plt.tight_layout()
plt.show()

## Photosynthetic outputs

The next example inspects selected photosynthetic readouts during the same irradiance transition, including CO2 assimilation, PSII efficiency, stomatal conductance, and Rubisco-limited rates.

In [ ]:
selected_outputs = ["A", "PhiII", "gsw", "Vc", "Vr", "VrJ"]

plt.figure(figsize=(8, 5))

for var in selected_outputs:
    if var in res.columns:
        y = res[var]
    elif f"obs_{var}" in res.columns:
        y = res[f"obs_{var}"]
    else:
        continue

    plt.plot(res.index / 60, y, label=var)

plt.xlabel("Time / min")
plt.ylabel("Model output")
plt.legend(frameon=False)
plt.tight_layout()
plt.show()

## Lightfleck-style response

This example simulates a short low-to-high-light transition, similar to the lightfleck analyses discussed by Morales et al. (2018).

In [ ]:
model = get_morales2018()

sim = Simulator(model)
sim.simulate_protocol(
    protocol=make_protocol(
        [
            (90, {"PAR": 50}),
            (90, {"PAR": 1000}),
        ]
    ),
    time_points_per_step=300,
)

fleck = sim.get_result().unwrap_or_err().get_combined()

fig, axs = plt.subplots(3, 1, figsize=(7, 8), sharex=True)

for ax, var in zip(axs, ["A", "fRuBP", "fRB"]):
    if var in fleck.columns:
        y = fleck[var]
    elif f"obs_{var}" in fleck.columns:
        y = fleck[f"obs_{var}"]
    else:
        continue

    ax.plot(fleck.index, y)
    ax.axvline(90, linestyle="--", linewidth=1)
    ax.set_ylabel(var)

axs[-1].set_xlabel("Time / s")
plt.tight_layout()
plt.show()